Next word prediction using LSTM

In [8]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

#load dataset
data=gutenberg.words('shakespeare-hamlet.txt')
with open('shakespeare-hamlet.txt', 'w') as f:
    f.write(' '.join(data))

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [9]:
#data preprocessing
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer #text to vector
from tensorflow.keras.preprocessing.sequence import pad_sequences #sentence length same for all
from sklearn.model_selection import train_test_split

#load the dataset
with open('shakespeare-hamlet.txt', 'r') as f:
    text = f.read().lower()
    

In [10]:
#Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text]) #applies directly creating indexes for word
total_words = len(tokenizer.word_index) + 1
total_words

4703

In [11]:
for i in range(1, 16):
    print(i, tokenizer.index_word.get(i))

1 the
2 and
3 '
4 to
5 of
6 i
7 you
8 a
9 my
10 it
11 in
12 that
13 ham
14 is
15 not


In [12]:
#create input sequences using list of tokens
#splitting each line and apply to tokenizer , the text converted to seq
inputssequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        inputssequences.append(n_gram)
print(inputssequences[:15])

[[1, 687], [1, 687, 5], [1, 687, 5, 50], [1, 687, 5, 50, 43], [1, 687, 5, 50, 43, 1854], [1, 687, 5, 50, 43, 1854, 1855], [1, 687, 5, 50, 43, 1854, 1855, 1856], [1, 687, 5, 50, 43, 1854, 1855, 1856, 1175], [1, 687, 5, 50, 43, 1854, 1855, 1856, 1175, 1857], [1, 687, 5, 50, 43, 1854, 1855, 1856, 1175, 1857, 1858], [1, 687, 5, 50, 43, 1854, 1855, 1856, 1175, 1857, 1858, 1859], [1, 687, 5, 50, 43, 1854, 1855, 1856, 1175, 1857, 1858, 1859, 62], [1, 687, 5, 50, 43, 1854, 1855, 1856, 1175, 1857, 1858, 1859, 62, 408], [1, 687, 5, 50, 43, 1854, 1855, 1856, 1175, 1857, 1858, 1859, 62, 408, 2], [1, 687, 5, 50, 43, 1854, 1855, 1856, 1175, 1857, 1858, 1859, 62, 408, 2, 1176]]


In [13]:
#Pad sequences to make them of equal length
max_sequence_len = max([len(x) for x in inputssequences])
inputssequences = np.array(pad_sequences(inputssequences, maxlen=max_sequence_len, padding='pre'))
inputssequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    5],
       [   0,    0,    0, ...,  687,    5,   50],
       ...,
       [   0,    0,    1, ...,    5,   50, 1043],
       [   0,    1,  687, ...,   50, 1043,    5],
       [   1,  687,    5, ..., 1043,    5,  203]], dtype=int32)

In [14]:
#create predictors and label
import tensorflow as tf
x,y=inputssequences[:,:-1],inputssequences[:,-1] #removed final word from each sequence and assigned to y


In [15]:
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    5],
       ...,
       [   0,    0,    1, ...,  687,    5,   50],
       [   0,    1,  687, ...,    5,   50, 1043],
       [   1,  687,    5, ...,   50, 1043,    5]], dtype=int32)

In [16]:
y

array([ 687,    5,   50, ..., 1043,    5,  203], dtype=int32)

In [17]:
y=tf.keras.utils.to_categorical(y, num_classes=total_words) #one hot encoding for y #output feature wrt to input feature
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [18]:
#split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((24799, 30999), (6200, 30999), (24799, 4703), (6200, 4703))

In [19]:
#Train our LSTM RNN model

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(
        input_dim=total_words,
        output_dim=16,
        input_length=max_sequence_len - 1
    ),
    LSTM(16),
    Dense(total_words, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 30999, 16)         75248     
                                                                 
 lstm (LSTM)                 (None, 16)                2112      
                                                                 
 dense (Dense)               (None, 4703)              79951     
                                                                 
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 30999, 16)         75248     
                                                                 
 lstm (LSTM)                 (None, 16)                2112      
                                                                 
 dense (Dense)               (None, 4703)              7

In [21]:
history = model.fit(
    x_train[:200],#to reduce the training time, we are using only 5000 samples for training
    y_train[:200],
    epochs=2,
    batch_size=32,
    verbose=1
)

Epoch 1/2
7/7 [==============================] - 458s 58s/step - loss: 8.4547 - accuracy: 0.0100
Epoch 2/2
7/7 [==============================] - 342s 48s/step - loss: 8.4435 - accuracy: 0.0550


In [22]:
model.save("next_word_model.keras")

In [23]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]

    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding='pre'
    )

    predicted = model.predict(token_list, verbose=0)

    predicted_word_index = np.argmax(predicted, axis=-1)[0]

    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word

    return None

In [24]:
text = input("Enter text: ")

predicted_word = predict_next_word(
    model,
    tokenizer,
    text,
    max_sequence_len
)

print("Next predicted word:", predicted_word)

Next predicted word: my
